In [1]:
import torch 
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2
from pathlib import Path

In [2]:
#Split the dataset into training and testing sets
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

# Path to the dataset
dataset_path = Path("../data/RealWaste")

# Collect image paths and their class labels
data = []

for class_folder in sorted(dataset_path.iterdir()):
    if class_folder.is_dir():
        for image_path in class_folder.iterdir():
            if image_path.is_file():
                data.append({
                    "image_path": str(image_path),
                    "label": class_folder.name
                })

df = pd.DataFrame(data)


train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    stratify=df["label"],
    random_state=42
)
print("\nTraining set size:", len(train_df))
print("Temporary set size:", len(temp_df))

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    stratify=temp_df["label"],
    random_state=42
)
print("Train:", len(train_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))


Training set size: 3326
Temporary set size: 1426
Train: 3326
Validation: 713
Test: 713


In [3]:
import torch
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights

weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)

print(model)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\Senadi Fernando/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:07<00:00, 2.92MB/s]


EfficientNet(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
      (2): SiLU(inplace=True)
    )
    (1): Sequential(
      (0): MBConv(
        (block): Sequential(
          (0): Conv2dNormActivation(
            (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
            (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, bias=True, track_running_stats=True)
            (2): SiLU(inplace=True)
          )
          (1): SqueezeExcitation(
            (avgpool): AdaptiveAvgPool2d(output_size=1)
            (fc1): Conv2d(32, 8, kernel_size=(1, 1), stride=(1, 1))
            (fc2): Conv2d(8, 32, kernel_size=(1, 1), stride=(1, 1))
            (activation): SiLU(inplace=True)
            (scale_activation): Sigmoid()
          )
          

In [4]:
#Replace the 1000 class classifier with a new classifier for 9 classes
num_classes = 9

model.classifier[1] = torch.nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

print(model.classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=9, bias=True)
)


In [5]:
from torchvision import transforms

# ImageNet normalization used by the pretrained MobileNetV2
normalize = transforms.Normalize(
    mean=[0.485, 0.456, 0.406],
    std=[0.229, 0.224, 0.225]
)

# Transformations for training images (Data augmenatation allowed as we dont want the model to overfit)
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(), #Convert the image to a tensor
    normalize
])

# Transformations for validation/test images
val_test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    normalize
])

print("Transforms created successfully.")

Transforms created successfully.


In [6]:
#create class labels 
class_names = sorted(train_df["label"].unique())

print(class_names)

['Cardboard', 'Food Organics', 'Glass', 'Metal', 'Miscellaneous Trash', 'Paper', 'Plastic', 'Textile Trash', 'Vegetation']


In [7]:
from torch.utils.data import Dataset
from PIL import Image


class WasteDataset(Dataset):

    def __init__(self, dataframe, transform=None):
        self.dataframe = dataframe.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        # Get image path and label from the DataFrame
        image_path = self.dataframe.iloc[index]["image_path"]
        label_name = self.dataframe.iloc[index]["label"]

        # Open the image
        image = Image.open(image_path).convert("RGB")

        # Convert the class name into a number
        label = class_names.index(label_name)

        # Apply the appropriate transformations
        if self.transform:
            image = self.transform(image)

        return image, label

In [8]:
train_dataset = WasteDataset(
    train_df,
    transform=train_transform
)

print("Number of training images:", len(train_dataset))

Number of training images: 3326


In [9]:
val_dataset = WasteDataset(
    val_df,
    transform=val_test_transform
)

test_dataset = WasteDataset(
    test_df,
    transform=val_test_transform
)

print("Training images:", len(train_dataset))
print("Validation images:", len(val_dataset))
print("Test images:", len(test_dataset))

Training images: 3326
Validation images: 713
Test images: 713


In [10]:
#Create data loaders for training, validation, and testing

from torch.utils.data import DataLoader

batch_size = 32

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size,
    shuffle=False
)



In [11]:
images, labels = next(iter(train_loader))

print("Batch image shape:", images.shape)
print("Batch labels shape:", labels.shape)

Batch image shape: torch.Size([32, 3, 224, 224])
Batch labels shape: torch.Size([32])


In [12]:
# Keep the pretrained weights frozen
# Freeze the pretrained EfficientNet-B0 backbone
for parameter in model.features.parameters():
    parameter.requires_grad = False

print("EfficientNet-B0 backbone frozen.")

EfficientNet-B0 backbone frozen.


In [13]:
criterion = torch.nn.CrossEntropyLoss()

print(criterion)
optimizer = torch.optim.Adam(
    model.classifier.parameters(),
    lr=0.001
)

print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [14]:
device = torch.device("cpu")

model = model.to(device)

print("Using device:", device)

Using device: cpu


In [15]:
images = images.to(device)
labels = labels.to(device)

outputs = model(images)

print("Output shape:", outputs.shape)

Output shape: torch.Size([32, 9])


In [16]:
loss = criterion(outputs, labels)

print("Loss:", loss.item())

Loss: 2.2542192935943604


In [17]:
images, labels = next(iter(train_loader))

images = images.to(device)
labels = labels.to(device)

print("Images:", images.shape)
print("Labels:", labels.shape)

Images: torch.Size([32, 3, 224, 224])
Labels: torch.Size([32])


In [18]:
optimizer.zero_grad()

In [19]:
loss.backward()

print("Backpropagation completed.")

Backpropagation completed.


In [20]:
optimizer.step()

print("Weights updated.")

Weights updated.


In [21]:
model.train()

running_loss = 0.0
correct = 0
total = 0

for images, labels in train_loader:

    images = images.to(device)
    labels = labels.to(device)

    # Forward pass
    outputs = model(images)

    # Calculate loss
    loss = criterion(outputs, labels)

    # Clear old gradients
    optimizer.zero_grad()

    # Backpropagation
    loss.backward()

    # Update weights
    optimizer.step()

    # Keep track of loss
    running_loss += loss.item()

    # Get predicted class
    _, predicted = torch.max(outputs, 1)

    # Count correct predictions
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

# Calculate average loss and accuracy
average_loss = running_loss / len(train_loader)
accuracy = 100 * correct / total

print("Training Loss:", average_loss)
print("Training Accuracy:", accuracy, "%")

Training Loss: 1.444156948190469
Training Accuracy: 54.35959110042093 %


In [22]:
num_epochs = 4

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Keep track of loss
        running_loss += loss.item()

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        # Count correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    average_loss = running_loss / len(train_loader)
    accuracy = 100 * correct / total

    print(
        f"Epoch {epoch + 2}/5 "
        f"- Loss: {average_loss:.4f} "
        f"- Accuracy: {accuracy:.2f}%"
    )

Epoch 2/5 - Loss: 0.9307 - Accuracy: 72.31%
Epoch 3/5 - Loss: 0.8079 - Accuracy: 74.71%
Epoch 4/5 - Loss: 0.7155 - Accuracy: 76.58%
Epoch 5/5 - Loss: 0.6729 - Accuracy: 77.60%


In [23]:
model.eval()

val_loss = 0.0
correct = 0
total = 0

with torch.no_grad():

    for images, labels in val_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Keep track of loss
        val_loss += loss.item()

        # Get predicted class
        _, predicted = torch.max(outputs, 1)

        # Count correct predictions
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

average_val_loss = val_loss / len(val_loader)
val_accuracy = 100 * correct / total

print("Validation Loss:", average_val_loss)
print("Validation Accuracy:", val_accuracy, "%")

Validation Loss: 0.6302927823170371
Validation Accuracy: 80.92566619915848 %


In [24]:
#5 more epochs of training
num_epochs = 5

for epoch in range(num_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        # Track loss
        running_loss += loss.item()

        # Track accuracy
        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    model.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            # Forward pass
            outputs = model(images)

            # Calculate loss
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            # Get predictions
            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(
        f"Epoch {epoch + 6}/10 "
        f"- Train Loss: {train_loss:.4f} "
        f"- Train Acc: {train_accuracy:.2f}% "
        f"- Val Loss: {val_loss:.4f} "
        f"- Val Acc: {val_accuracy:.2f}%"
    )

Epoch 6/10 - Train Loss: 0.6218 - Train Acc: 80.49% - Val Loss: 0.5994 - Val Acc: 79.52%
Epoch 7/10 - Train Loss: 0.6064 - Train Acc: 79.98% - Val Loss: 0.5808 - Val Acc: 81.21%
Epoch 8/10 - Train Loss: 0.5699 - Train Acc: 80.73% - Val Loss: 0.5597 - Val Acc: 81.21%
Epoch 9/10 - Train Loss: 0.5351 - Train Acc: 83.10% - Val Loss: 0.5412 - Val Acc: 82.89%
Epoch 10/10 - Train Loss: 0.5254 - Train Acc: 83.40% - Val Loss: 0.5358 - Val Acc: 83.03%


In [25]:
for i, layer in enumerate(model.features):
    print(i, type(layer).__name__)

0 Conv2dNormActivation
1 Sequential
2 Sequential
3 Sequential
4 Sequential
5 Sequential
6 Sequential
7 Sequential
8 Conv2dNormActivation


In [26]:
# Unfreeze the deeper EfficientNetB0 layers
for parameter in model.features[6:].parameters():
    parameter.requires_grad = True



In [27]:
# Check which parameters are trainable

for name, parameter in model.named_parameters():
    if parameter.requires_grad:
        print(name)

features.6.0.block.0.0.weight
features.6.0.block.0.1.weight
features.6.0.block.0.1.bias
features.6.0.block.1.0.weight
features.6.0.block.1.1.weight
features.6.0.block.1.1.bias
features.6.0.block.2.fc1.weight
features.6.0.block.2.fc1.bias
features.6.0.block.2.fc2.weight
features.6.0.block.2.fc2.bias
features.6.0.block.3.0.weight
features.6.0.block.3.1.weight
features.6.0.block.3.1.bias
features.6.1.block.0.0.weight
features.6.1.block.0.1.weight
features.6.1.block.0.1.bias
features.6.1.block.1.0.weight
features.6.1.block.1.1.weight
features.6.1.block.1.1.bias
features.6.1.block.2.fc1.weight
features.6.1.block.2.fc1.bias
features.6.1.block.2.fc2.weight
features.6.1.block.2.fc2.bias
features.6.1.block.3.0.weight
features.6.1.block.3.1.weight
features.6.1.block.3.1.bias
features.6.2.block.0.0.weight
features.6.2.block.0.1.weight
features.6.2.block.0.1.bias
features.6.2.block.1.0.weight
features.6.2.block.1.1.weight
features.6.2.block.1.1.bias
features.6.2.block.2.fc1.weight
features.6.2.blo

In [28]:
# Use a smaller learning rate for fine tuning
optimizer = torch.optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=0.0001
)


In [29]:
num_epochs = 5

for epoch in range(num_epochs):

    # Training
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Clear old gradients
        optimizer.zero_grad()

        # Backpropagation
        loss.backward()

        # Update weights
        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / len(train_loader)
    train_accuracy = 100 * correct / total

    # Validation
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():

        for images, labels in val_loader:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / len(val_loader)
    val_accuracy = 100 * correct / total

    print(
        f"Fine-tuning Epoch {epoch + 1}/5 "
        f"- Train Loss: {train_loss:.4f} "
        f"- Train Acc: {train_accuracy:.2f}% "
        f"- Val Loss: {val_loss:.4f} "
        f"- Val Acc: {val_accuracy:.2f}%"
    )

Fine-tuning Epoch 1/5 - Train Loss: 0.4714 - Train Acc: 84.28% - Val Loss: 0.4171 - Val Acc: 86.26%
Fine-tuning Epoch 2/5 - Train Loss: 0.2896 - Train Acc: 90.89% - Val Loss: 0.3579 - Val Acc: 87.24%
Fine-tuning Epoch 3/5 - Train Loss: 0.2033 - Train Acc: 94.59% - Val Loss: 0.3216 - Val Acc: 88.78%
Fine-tuning Epoch 4/5 - Train Loss: 0.1330 - Train Acc: 96.54% - Val Loss: 0.2922 - Val Acc: 89.48%
Fine-tuning Epoch 5/5 - Train Loss: 0.1169 - Train Acc: 97.08% - Val Loss: 0.2802 - Val Acc: 89.90%


In [30]:
# Save the fine tuned EfficientNet-B0 model

torch.save(
    model.state_dict(),
    "efficientnetb0_realwaste_best.pth"
)



In [31]:
# Evaluate EfficientNet-B0 on the test set

model.eval()

test_loss = 0.0
correct = 0
total = 0

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss = test_loss / total
test_accuracy = 100 * correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.3624
Test Accuracy: 87.24%


In [32]:


model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())



In [33]:
from sklearn.metrics import classification_report

report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
)

print(report)

                     precision    recall  f1-score   support

          Cardboard       0.90      0.88      0.89        69
      Food Organics       0.93      0.90      0.92        62
              Glass       0.93      0.83      0.87        63
              Metal       0.80      0.86      0.83       118
Miscellaneous Trash       0.75      0.75      0.75        75
              Paper       0.93      0.88      0.90        75
            Plastic       0.87      0.88      0.87       138
      Textile Trash       0.88      0.96      0.92        48
         Vegetation       0.97      0.95      0.96        65

           accuracy                           0.87       713
          macro avg       0.88      0.88      0.88       713
       weighted avg       0.87      0.87      0.87       713



In [34]:
#create a confusion matrix
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(all_labels, all_predictions)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print(cm_df)

                     Cardboard  Food Organics  Glass  Metal  \
Cardboard                   61              0      0      5   
Food Organics                0             56      0      0   
Glass                        0              0     52      5   
Metal                        1              0      0    102   
Miscellaneous Trash          2              1      1      4   
Paper                        4              1      0      2   
Plastic                      0              0      3     10   
Textile Trash                0              0      0      0   
Vegetation                   0              2      0      0   

                     Miscellaneous Trash  Paper  Plastic  Textile Trash  \
Cardboard                              0      1        1              1   
Food Organics                          4      0        0              1   
Glass                                  3      0        3              0   
Metal                                  6      1        8             

In [35]:
# Continue fine-tuning for 5 more epochs

num_additional_epochs = 5

for epoch in range(num_additional_epochs):

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    train_loss = running_loss / total
    train_accuracy = 100 * correct / total

    # Validation
    model.eval()

    val_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    val_loss = val_loss / total
    val_accuracy = 100 * correct / total

    print(
        f"Additional Epoch {epoch+1}/5 - "
        f"Train Loss: {train_loss:.4f} - "
        f"Train Acc: {train_accuracy:.2f}% - "
        f"Val Loss: {val_loss:.4f} - "
        f"Val Acc: {val_accuracy:.2f}%"
    )

Additional Epoch 1/5 - Train Loss: 0.0874 - Train Acc: 97.84% - Val Loss: 0.2839 - Val Acc: 90.04%
Additional Epoch 2/5 - Train Loss: 0.0799 - Train Acc: 97.96% - Val Loss: 0.2690 - Val Acc: 91.02%
Additional Epoch 3/5 - Train Loss: 0.0613 - Train Acc: 98.41% - Val Loss: 0.2716 - Val Acc: 90.74%
Additional Epoch 4/5 - Train Loss: 0.0517 - Train Acc: 98.77% - Val Loss: 0.2891 - Val Acc: 89.90%
Additional Epoch 5/5 - Train Loss: 0.0484 - Train Acc: 98.65% - Val Loss: 0.2774 - Val Acc: 90.32%


In [36]:


torch.save(
    model.state_dict(),
    "efficientnetb0_realwaste_1.pth"
)



In [37]:
model.eval()

test_loss = 0.0
correct = 0
total = 0

criterion = torch.nn.CrossEntropyLoss()

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        loss = criterion(outputs, labels)
        test_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)
        correct += (predicted == labels).sum().item()

test_loss = test_loss / total
test_accuracy = 100 * correct / total

print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_accuracy:.2f}%")

Test Loss: 0.3521
Test Accuracy: 88.92%


In [38]:
model.eval()

all_labels = []
all_predictions = []

with torch.no_grad():
    for images, labels in test_loader:
        outputs = model(images)

        _, predicted = torch.max(outputs, 1)

        all_labels.extend(labels.cpu().numpy())
        all_predictions.extend(predicted.cpu().numpy())


In [39]:
from sklearn.metrics import classification_report

report = classification_report(
    all_labels,
    all_predictions,
    target_names=class_names
)

print(report)

                     precision    recall  f1-score   support

          Cardboard       0.92      0.94      0.93        69
      Food Organics       0.93      0.90      0.92        62
              Glass       0.93      0.84      0.88        63
              Metal       0.87      0.89      0.88       118
Miscellaneous Trash       0.79      0.80      0.79        75
              Paper       0.94      0.88      0.91        75
            Plastic       0.85      0.89      0.87       138
      Textile Trash       0.90      0.92      0.91        48
         Vegetation       0.97      0.95      0.96        65

           accuracy                           0.89       713
          macro avg       0.90      0.89      0.89       713
       weighted avg       0.89      0.89      0.89       713



In [40]:
#create a confusion matrix
from sklearn.metrics import confusion_matrix
import pandas as pd

cm = confusion_matrix(all_labels, all_predictions)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print(cm_df)

                     Cardboard  Food Organics  Glass  Metal  \
Cardboard                   65              0      0      0   
Food Organics                0             56      0      0   
Glass                        0              0     53      3   
Metal                        1              0      0    105   
Miscellaneous Trash          1              1      1      3   
Paper                        4              1      0      1   
Plastic                      0              0      3      8   
Textile Trash                0              0      0      1   
Vegetation                   0              2      0      0   

                     Miscellaneous Trash  Paper  Plastic  Textile Trash  \
Cardboard                              0      1        3              0   
Food Organics                          4      0        0              1   
Glass                                  2      1        4              0   
Metal                                  4      1        7             

In [2]:
import os
import torch
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights


weights = EfficientNet_B0_Weights.DEFAULT

model = efficientnet_b0(weights=weights)


num_classes = 9

model.classifier[1] = torch.nn.Linear(
    model.classifier[1].in_features,
    num_classes
)

model.load_state_dict(
    torch.load(
        "efficientnetb0_realwaste_1.pth",
        map_location="cpu"
    )
)


model.eval()



total_params = sum(
    p.numel()
    for p in model.parameters()
)


model_size_mb = os.path.getsize(
    "efficientnetb0_realwaste_1.pth"
) / (1024 ** 2)



print("EfficientNet-B0 loaded successfully!")
print()
print("Total parameters:", total_params)
print(f"Model size: {model_size_mb:.2f} MB")

EfficientNet-B0 loaded successfully!

Total parameters: 4019077
Model size: 15.62 MB
